In [ ]:
!pip install xgboost lightgbm pytorch-tabnet optuna imbalanced-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.6 MB/s eta 0:00:00


In [ ]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, log_loss,
    classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
)
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier
import lightgbm as lgb
from pytorch_tabnet.tab_model import TabNetClassifier

from imblearn.over_sampling import BorderlineSMOTE, RandomOverSampler
from collections import Counter

In [ ]:
df = pd.read_csv("final_5.csv")
print("Shape:", df.shape)

EYE_TARGET  = "Predicted_Eye_Color"
HAIR_TARGET = "Predicted_Hair_Color"
SKIN_TARGET = "Predicted_Skin_Color"

snp_cols = [c for c in df.columns if c.startswith("rs")]
print(f"SNP columns: {len(snp_cols)}")

for t in [EYE_TARGET, HAIR_TARGET, SKIN_TARGET]:
    print(f"\n{t}:")
    print(df[t].value_counts())

Shape: (10317, 60)
SNP columns: 40

Predicted_Eye_Color:
Predicted_Eye_Color
Brown           6614
Blue            3697
Intermediate       6
Name: count, dtype: int64

Predicted_Hair_Color:
Predicted_Hair_Color
Brown    5075
Black    2500
Blond    2473
Red       269
Name: count, dtype: int64

Predicted_Skin_Color:
Predicted_Skin_Color
Intermediate     4318
Pale             2904
Dark             1270
DarkToBlack      1202
Dark to Black     548
Very Pale          49
VeryPale           26
Name: count, dtype: int64


In [ ]:
nan_count = int(df[snp_cols].isna().sum().sum())
if nan_count > 0:
    print(f"Found {nan_count} NaN values - imputing with column mode...")
    for c in snp_cols:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].mode().iloc[0])
else:
    print("No NaN values found.")

var_selector = VarianceThreshold(threshold=0.01)
var_selector.fit(df[snp_cols].values)
selected_snps = [snp_cols[i] for i in range(len(snp_cols)) if var_selector.get_support()[i]]
removed_snps  = [c for c in snp_cols if c not in selected_snps]
print(f"\nSelected SNPs: {len(selected_snps)}")
print(f"Removed (low variance): {removed_snps}")

Found 15 NaN values - imputing with column mode...

Selected SNPs: 38
Removed (low variance): ['rs11547464_A', 'rs201326893_A']


## Class Merge: Pale + VeryPale

Merge **VeryPale** into **Pale** to reduce class imbalance.

In [ ]:
def _clean_skin(v):
    s = str(v).strip().replace(" ", "").lower()
    mapping = {
        "darktoblack": "DarkToBlack", "dark": "Dark",
        "intermediate": "Intermediate", "pale": "Pale", "verypale": "VeryPale",
    }
    return mapping.get(s, s)

SKIN_TARGET_MERGED = "Predicted_Skin_Color_merged"
df[SKIN_TARGET_MERGED] = df[SKIN_TARGET].map(_clean_skin).replace("VeryPale", "Pale")

print("--- Before merge ---")
print(df[SKIN_TARGET].value_counts())
print("\n--- After merge ---")
print(df[SKIN_TARGET_MERGED].value_counts())

--- Before merge ---
Predicted_Skin_Color
Intermediate     4318
Pale             2904
Dark             1270
DarkToBlack      1202
Dark to Black     548
Very Pale          49
VeryPale           26
Name: count, dtype: int64

--- After merge ---
Predicted_Skin_Color_merged
Intermediate    4318
Pale            2979
DarkToBlack     1750
Dark            1270
Name: count, dtype: int64


## [NEW #9] Gene-Group Feature Engineering

Aggregate features from known pigmentation gene groups (was missing from old notebook):
- **MC1R**: red hair / fair skin
- **HERC2/OCA2**: eye color (HERC2_OCA2_mean was top-13 for eye in SERVER2)
- **SLC24A5/SLC45A2**: skin pigmentation
- **TYR**: eye/skin
- **Interactions**: MC1R x HERC2, MC1R x SLC

In [ ]:
gene_groups = {
    "MC1R":       ["rs1805005","rs1805006","rs1805007","rs1805008","rs1805009",
                   "rs2228479","rs11547464","rs885479","rs1110400"],
    "HERC2_OCA2": ["rs12913832","rs1129038","rs1470608","rs1800407","rs12896399"],
    "SLC":        ["rs1426654","rs16891982","rs28777"],
    "TYR":        ["rs1042602","rs1393350","rs1126809"],
}

gene_group_cols = []
for group_name, rs_ids in gene_groups.items():
    cols = [s for s in selected_snps if any(rs in s for rs in rs_ids)]
    if len(cols) >= 2:
        df[f"{group_name}_sum"]  = df[cols].sum(axis=1)
        df[f"{group_name}_mean"] = df[cols].mean(axis=1)
        gene_group_cols += [f"{group_name}_sum", f"{group_name}_mean"]
        print(f"{group_name}: {len(cols)} SNPs -> sum + mean")
    elif len(cols) == 1:
        print(f"{group_name}: 1 SNP only, skipping aggregate")

if "MC1R_sum" in df.columns and "HERC2_OCA2_sum" in df.columns:
    df["MC1R_x_HERC2"] = df["MC1R_sum"] * df["HERC2_OCA2_sum"]
    gene_group_cols.append("MC1R_x_HERC2")
if "MC1R_sum" in df.columns and "SLC_sum" in df.columns:
    df["MC1R_x_SLC"] = df["MC1R_sum"] * df["SLC_sum"]
    gene_group_cols.append("MC1R_x_SLC")

all_feature_cols = selected_snps + gene_group_cols
print(f"\nTotal features: {len(all_feature_cols)} ({len(selected_snps)} SNPs + {len(gene_group_cols)} gene-group)")

MC1R: 8 SNPs -> sum + mean
HERC2_OCA2: 5 SNPs -> sum + mean
SLC: 3 SNPs -> sum + mean
TYR: 3 SNPs -> sum + mean

Total features: 48 (38 SNPs + 10 gene-group)


## Feature Selection (RF Importance per Trait)

Now selects from SNPs **+ gene-group features** (expanded pool).

In [ ]:
targets_fs = {
    "Eye Color":  EYE_TARGET,
    "Hair Color": HAIR_TARGET,
    "Skin Color": SKIN_TARGET_MERGED,
}
target_n_features = {"Eye Color": 15, "Hair Color": 35, "Skin Color": 30}
trait_feature_cols = {}

for label, target_col in targets_fs.items():
    n = target_n_features[label]
    X_fs = df[all_feature_cols]
    y_fs = df[target_col].astype(str)
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_fs, y_fs)
    importances = pd.Series(rf.feature_importances_, index=all_feature_cols)
    top_n = importances.nlargest(n).index.tolist()
    trait_feature_cols[label] = top_n

    print(f"\n--- {label} (Top {n}) ---")
    for feat in top_n:
        print(f"  {feat}: {importances[feat]:.4f}")


--- Eye Color (Top 15) ---
  rs12913832_T: 0.2433
  rs1129038_G: 0.2107
  HERC2_OCA2_sum: 0.1271
  HERC2_OCA2_mean: 0.1087
  rs1667394_C: 0.0631
  rs2238289_C: 0.0331
  rs1470608_A: 0.0307
  MC1R_x_HERC2: 0.0227
  rs12896399_T: 0.0203
  SLC_sum: 0.0162
  rs6497292_C: 0.0159
  rs16891982_C: 0.0158
  SLC_mean: 0.0156
  rs1800407_A: 0.0093
  rs1426654_G: 0.0086

--- Hair Color (Top 35) ---
  rs16891982_C: 0.1133
  rs12203592_T: 0.0917
  rs12913832_T: 0.0735
  rs1129038_G: 0.0679
  SLC_mean: 0.0632
  SLC_sum: 0.0625
  HERC2_OCA2_sum: 0.0406
  HERC2_OCA2_mean: 0.0342
  rs28777_C: 0.0328
  rs1667394_C: 0.0227
  rs1426654_G: 0.0214
  MC1R_x_SLC: 0.0190
  MC1R_x_HERC2: 0.0183
  rs683_G: 0.0152
  rs4959270_A: 0.0150
  rs1805008_T: 0.0147
  rs8051733_C: 0.0143
  TYR_sum: 0.0142
  TYR_mean: 0.0136
  MC1R_sum: 0.0129
  rs1470608_A: 0.0125
  rs2238289_C: 0.0121
  MC1R_mean: 0.0121
  rs1805007_T: 0.0120
  rs3114908_T: 0.0119
  rs6119471_C: 0.0116
  rs1042602_T: 0.0116
  rs12821256_G: 0.0114
  rs180

In [ ]:
def compute_metrics(y_true, y_pred, y_proba, n_classes):
    return {
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Macro_F1": round(f1_score(y_true, y_pred, average="macro"), 4),
        "Log_Loss": round(log_loss(y_true, y_proba, labels=list(range(n_classes))), 4),
    }

In [ ]:
class TabNetWrapper:
    _estimator_type = "classifier"

    def __init__(self, **kwargs):
        self.kwargs = kwargs
        self.model = None
        self.classes_ = None

    def fit(self, X, y, **fit_params):
        self.classes_ = np.unique(y)
        self.model = TabNetClassifier(**self.kwargs)
        self.model.fit(X.astype(np.float32), y, **fit_params)
        return self

    def predict(self, X):
        return self.model.predict(X.astype(np.float32))

    def predict_proba(self, X):
        return self.model.predict_proba(X.astype(np.float32))

    def get_params(self, deep=True):
        return self.kwargs.copy()

    def set_params(self, **params):
        self.kwargs.update(params)
        return self

In [ ]:
def optuna_xgb(X_train, y_train, n_classes, n_trials=15):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 600),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
        mdl = XGBClassifier(device='cuda',
            **params, objective="multi:softprob", num_class=n_classes,
            random_state=42, verbosity=0, n_jobs=-1)
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        scores = []
        for tr, va in skf.split(X_train, y_train):
            mdl.fit(X_train[tr], y_train[tr])
            scores.append(f1_score(y_train[va], mdl.predict(X_train[va]), average="macro"))
        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    print(f"    XGB best CV-F1={study.best_value:.4f}")
    return study.best_params


def optuna_lgbm(X_train, y_train, n_classes, n_trials=15):
    obj = "multiclass" if n_classes > 2 else "binary"
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 600),
            "num_leaves": trial.suggest_int("num_leaves", 15, 63),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
        mdl = lgb.LGBMClassifier(device_type='gpu',
            **params, objective=obj,
            num_class=n_classes if obj == "multiclass" else None,
            class_weight="balanced" if obj == "multiclass" else None,
            random_state=42, n_jobs=-1, verbose=-1)
        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        scores = []
        for tr, va in skf.split(X_train, y_train):
            mdl.fit(X_train[tr], y_train[tr])
            scores.append(f1_score(y_train[va], mdl.predict(X_train[va]), average="macro"))
        return np.mean(scores)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    print(f"    LGBM best CV-F1={study.best_value:.4f}")
    return study.best_params


def apply_smote(X, y):
    min_count = min(Counter(y).values())
    if min_count < 2:
        print("    Oversampling: skipped (class with <2 samples)")
        return X, y
    if min_count <= 5:
        ros = RandomOverSampler(random_state=42)
        X_r, y_r = ros.fit_resample(X, y)
        print(f"    RandomOverSampler: {len(y)} -> {len(y_r)}")
        return X_r, y_r
    try:
        k = min(5, min_count - 1)
        sm = BorderlineSMOTE(random_state=42, k_neighbors=k)
        X_r, y_r = sm.fit_resample(X, y)
        print(f"    BorderlineSMOTE: {len(y)} -> {len(y_r)}")
        return X_r, y_r
    except Exception:
        ros = RandomOverSampler(random_state=42)
        X_r, y_r = ros.fit_resample(X, y)
        print(f"    Fallback RandomOverSampler: {len(y)} -> {len(y_r)}")
        return X_r, y_r

## Train & Evaluate All Models per Trait

**8 models** (was 6+duplicates): LogReg, LightGBM (Optuna), XGBoost (Optuna), HistGB, SVC, RF, TabNet, Stacking.

Key changes from old notebook:
- [FIX #1] `multi_class='multinomial'` removed from LogReg (crashes sklearn 1.7+)
- [FIX #2] Single XGBoost (was duplicate xgb_m + xgb_fold)
- [FIX #3] Proper train/calib/test split (no data leakage)
- [FIX #6] TabNet weights=0 (was passing per-sample weights incorrectly)
- [FIX #11] SVC calibrated properly via CalibratedClassifierCV (no probability=True hack)
- [NEW] All models calibrated with sigmoid/isotonic on held-out set
- [NEW #B] Stacking ensemble (LGBM+XGB+HistGB+RF -> LogReg meta-learner)
- [NEW] RandomForest added as 6th individual model

In [ ]:
def train_eval_trait(trait_name, target_col, feature_cols, df_data):
    le = LabelEncoder()
    y_all = le.fit_transform(df_data[target_col].astype(str))
    classes = le.classes_
    n_classes = len(classes)
    X_all = df_data[feature_cols].values

    print(f"\n{'='*60}")
    print(f"  {trait_name}  |  {n_classes} classes: {list(classes)}  |  {len(feature_cols)} features")
    print(f"{'='*60}")

    X_tv, X_test, y_tv, y_test = train_test_split(
        X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
    X_train, X_calib, y_train, y_calib = train_test_split(
        X_tv, y_tv, test_size=0.125, random_state=42, stratify=y_tv)
    print(f"  Split: train={len(y_train)}, calib={len(y_calib)}, test={len(y_test)}")

    X_sm, y_sm = apply_smote(X_train, y_train)

    X_tr, X_es, y_tr, y_es = train_test_split(
        X_sm, y_sm, test_size=0.15, random_state=42, stratify=y_sm)

    min_cal = min(Counter(y_calib).values())
    cal_method = "sigmoid" if min_cal < 10 else "isotonic"
    print(f"  Calibration: {cal_method} (min calib class={min_cal})")

    models, metrics_dict, preds = {}, {}, {}

    def _register(name, model, X_te=X_test):
        pred = model.predict(X_te)
        proba = model.predict_proba(X_te)
        models[name] = model
        metrics_dict[name] = compute_metrics(y_test, pred, proba, n_classes)
        preds[name] = pred
        m = metrics_dict[name]
        print(f"    {name:<12s}  Acc={m['Accuracy']:.4f}  F1={m['Macro_F1']:.4f}  Loss={m['Log_Loss']:.4f}")

    print("  [1/8] LogisticRegression")
    lr = Pipeline([("scaler", StandardScaler()),
                   ("clf", LogisticRegression(solver="lbfgs", max_iter=2000, class_weight="balanced"))])
    lr.fit(X_sm, y_sm)
    lr_cal = CalibratedClassifierCV(lr, method=cal_method, cv="prefit")
    lr_cal.fit(X_calib, y_calib)
    _register("LogReg", lr_cal)

    print("  [2/8] LightGBM (Optuna)")
    lgbm_params = optuna_lgbm(X_sm, y_sm, n_classes, n_trials=15)
    obj_type = "multiclass" if n_classes > 2 else "binary"
    lgbm_m = lgb.LGBMClassifier(device_type='gpu',
        **lgbm_params, objective=obj_type,
        num_class=n_classes if obj_type == "multiclass" else None,
        class_weight="balanced" if obj_type == "multiclass" else None,
        random_state=42, n_jobs=-1, verbose=-1)
    lgbm_m.fit(X_tr, y_tr, eval_set=[(X_es, y_es)],
               callbacks=[lgb.early_stopping(stopping_rounds=25, verbose=False)])
    lgbm_cal = CalibratedClassifierCV(lgbm_m, method=cal_method, cv="prefit")
    lgbm_cal.fit(X_calib, y_calib)
    _register("LGBM", lgbm_cal)

    print("  [3/8] XGBoost (Optuna)")
    xgb_params = optuna_xgb(X_sm, y_sm, n_classes, n_trials=15)
    xgb_m = XGBClassifier(device='cuda',
        **xgb_params, objective="multi:softprob", num_class=n_classes,
        random_state=42, verbosity=0, n_jobs=-1)
    xgb_m.fit(X_sm, y_sm)
    xgb_cal = CalibratedClassifierCV(xgb_m, method=cal_method, cv="prefit")
    xgb_cal.fit(X_calib, y_calib)
    _register("XGB", xgb_cal)

    print("  [4/8] HistGradientBoosting")
    hist_m = HistGradientBoostingClassifier(
        max_iter=300, max_depth=5, learning_rate=0.05,
        random_state=42, class_weight="balanced")
    hist_m.fit(X_sm, y_sm)
    hist_cal = CalibratedClassifierCV(hist_m, method=cal_method, cv="prefit")
    hist_cal.fit(X_calib, y_calib)
    _register("HistGB", hist_cal)

    print("  [5/8] SVC")
    svc_pipe = Pipeline([("scaler", StandardScaler()),
                         ("clf", SVC(kernel="rbf", C=1.0, gamma="scale",
                                     random_state=42, class_weight="balanced"))])
    svc_pipe.fit(X_sm, y_sm)
    svc_cal = CalibratedClassifierCV(svc_pipe, method=cal_method, cv="prefit")
    svc_cal.fit(X_calib, y_calib)
    _register("SVC", svc_cal)

    print("  [6/8] RandomForest")
    rf_m = RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1)
    rf_m.fit(X_sm, y_sm)
    rf_cal = CalibratedClassifierCV(rf_m, method=cal_method, cv="prefit")
    rf_cal.fit(X_calib, y_calib)
    _register("RF", rf_cal)

    print("  [7/8] TabNet")
    tab_m = TabNetWrapper(
        n_d=32, n_a=32, n_steps=5, gamma=1.5,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size": 10, "gamma": 0.9},
        verbose=0)
    tab_m.fit(X_sm, y_sm,
              weights=0,
              max_epochs=60, patience=10,
              eval_set=[(X_es.astype(np.float32), y_es)])
    tab_cal = CalibratedClassifierCV(tab_m, method=cal_method, cv="prefit")
    tab_cal.fit(X_calib, y_calib)
    _register("TabNet", tab_cal)

    print("  [8/8] Stacking Ensemble")
    stack_est = [
        ("lgbm", lgb.LGBMClassifier(device_type='gpu',
            **lgbm_params, objective=obj_type,
            num_class=n_classes if obj_type == "multiclass" else None,
            class_weight="balanced" if obj_type == "multiclass" else None,
            random_state=42, n_jobs=-1, verbose=-1)),
        ("xgb", XGBClassifier(device='cuda',
            **xgb_params, objective="multi:softprob", num_class=n_classes,
            random_state=42, verbosity=0, n_jobs=-1)),
        ("hist", HistGradientBoostingClassifier(
            max_iter=300, max_depth=5, learning_rate=0.05,
            random_state=42, class_weight="balanced")),
        ("rf", RandomForestClassifier(
            n_estimators=300, random_state=42, class_weight="balanced", n_jobs=-1)),
    ]
    stacker = StackingClassifier(
        estimators=stack_est,
        final_estimator=LogisticRegression(max_iter=1000, class_weight="balanced"),
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        stack_method="predict_proba", n_jobs=-1)
    stacker.fit(X_sm, y_sm)
    stack_cal = CalibratedClassifierCV(stacker, method=cal_method, cv="prefit")
    stack_cal.fit(X_calib, y_calib)
    _register("Stacking", stack_cal)

    print(f"\n  {'Model':<12s} {'Accuracy':>10s} {'Macro_F1':>10s} {'Log_Loss':>10s}")
    print(f"  {'-'*44}")
    for name in models:
        m = metrics_dict[name]
        print(f"  {name:<12s} {m['Accuracy']:>10.4f} {m['Macro_F1']:>10.4f} {m['Log_Loss']:>10.4f}")

    return {
        "label_encoder": le,
        "classes": classes,
        "models": models,
        "metrics": metrics_dict,
        "y_test": y_test,
        "predictions": preds,
        "feature_cols": feature_cols,
        "optuna_params": {"xgb": xgb_params, "lgbm": lgbm_params},
    }

In [ ]:
eye_res  = train_eval_trait("Eye Color",  EYE_TARGET,         trait_feature_cols["Eye Color"],  df)
hair_res = train_eval_trait("Hair Color", HAIR_TARGET,        trait_feature_cols["Hair Color"], df)
skin_res = train_eval_trait("Skin Color", SKIN_TARGET_MERGED, trait_feature_cols["Skin Color"], df)


  Eye Color  |  3 classes: ['Blue', 'Brown', 'Intermediate']  |  15 features
  Split: train=7221, calib=1032, test=2064
    RandomOverSampler: 7221 -> 13890
  Calibration: sigmoid (min calib class=1)
  [1/8] LogisticRegression
    LogReg        Acc=0.9956  F1=0.6637  Loss=0.0738
  [2/8] LightGBM (Optuna)
    LGBM best CV-F1=0.9930
    LGBM          Acc=0.9913  F1=0.6624  Loss=0.0350
  [3/8] XGBoost (Optuna)
    XGB best CV-F1=0.9928
    XGB           Acc=0.9913  F1=0.6624  Loss=0.0350
  [4/8] HistGradientBoosting
    HistGB        Acc=0.9952  F1=0.6634  Loss=0.0249
  [5/8] SVC
    SVC           Acc=0.9908  F1=0.7137  Loss=0.0348
  [6/8] RandomForest
    RF            Acc=0.9913  F1=0.6619  Loss=0.0395
  [7/8] TabNet

Early stopping occurred at epoch 28 with best_epoch = 18 and best_val_0_accuracy = 0.99328
    TabNet        Acc=0.9893  F1=0.7436  Loss=0.0397
  [8/8] Stacking Ensemble
    Stacking      Acc=0.9922  F1=0.6626  Loss=0.0305

  Model          Accuracy   Macro_F1   Log_Loss


In [ ]:
def choose_best_model(res_dict):
    best_name, best_m = None, None
    for name, m in res_dict["metrics"].items():
        if best_m is None \
           or m["Macro_F1"] > best_m["Macro_F1"] \
           or (m["Macro_F1"] == best_m["Macro_F1"] and m["Log_Loss"] < best_m["Log_Loss"]):
            best_name, best_m = name, m
    return best_name, res_dict["models"][best_name]


eye_choice,  eye_final_model  = choose_best_model(eye_res)
hair_choice, hair_final_model = choose_best_model(hair_res)
skin_choice, skin_final_model = choose_best_model(skin_res)

print("="*50)
print("FINAL MODEL CHOICES")
print("="*50)
for label, choice, res in [("Eye", eye_choice, eye_res),
                            ("Hair", hair_choice, hair_res),
                            ("Skin", skin_choice, skin_res)]:
    m = res["metrics"][choice]
    print(f"  {label:<6s} -> {choice:<12s}  F1={m['Macro_F1']:.4f}  Loss={m['Log_Loss']:.4f}")

In [ ]:

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, log_loss

N_BOOTSTRAP = 100
CI_LEVEL = 0.95
RANDOM_STATE = 42

ci_traits = {
    "Eye": {
        "res": eye_res,
        "target_col": EYE_TARGET,
        "best_name": eye_choice,
        "best_model": eye_final_model,
    },
    "Hair": {
        "res": hair_res,
        "target_col": HAIR_TARGET,
        "best_name": hair_choice,
        "best_model": hair_final_model,
    },
    "Skin": {
        "res": skin_res,
        "target_col": SKIN_TARGET_MERGED,
        "best_name": skin_choice,
        "best_model": skin_final_model,
    },
}


def rebuild_heldout_test_set(res, target_col):
    """
    Rebuilds the exact same 20% held-out test split used during training.
    This matches the train_eval_trait() split:
    train/calib/test = 70% / 10% / 20%
    """
    le = res["label_encoder"]
    feature_cols = res["feature_cols"]

    X_all = df[feature_cols].values
    y_all = le.transform(df[target_col].astype(str))

    _, X_test, _, y_test = train_test_split(
        X_all,
        y_all,
        test_size=0.2,
        random_state=42,
        stratify=y_all
    )

    return X_test, y_test


def stratified_bootstrap_indices(y, rng):
    """
    Draws bootstrap samples while preserving the class distribution.
    This is important because the Intermediate eye class is extremely rare.
    """
    y = np.asarray(y)
    all_indices = np.arange(len(y))
    sampled_indices = []

    for cls in np.unique(y):
        cls_indices = all_indices[y == cls]
        sampled_cls_indices = rng.choice(
            cls_indices,
            size=len(cls_indices),
            replace=True
        )
        sampled_indices.extend(sampled_cls_indices)

    sampled_indices = np.array(sampled_indices)
    rng.shuffle(sampled_indices)
    return sampled_indices


def confidence_interval(values, level=0.95):
    alpha = (1 - level) / 2
    lower = np.percentile(values, 100 * alpha)
    upper = np.percentile(values, 100 * (1 - alpha))
    return lower, upper


def evaluate_metrics(y_true, y_pred, y_proba, n_classes):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_F1": f1_score(
            y_true,
            y_pred,
            average="macro",
            labels=list(range(n_classes)),
            zero_division=0
        ),
        "Log_Loss": log_loss(
            y_true,
            y_proba,
            labels=list(range(n_classes))
        )
    }


bootstrap_rows = []
summary_rows = []

rng = np.random.default_rng(RANDOM_STATE)

for trait_name, cfg in ci_traits.items():
    res = cfg["res"]
    model = cfg["best_model"]
    best_name = cfg["best_name"]
    classes = res["classes"]
    n_classes = len(classes)

    X_test, y_test = rebuild_heldout_test_set(res, cfg["target_col"])

    y_pred_full = model.predict(X_test)
    y_proba_full = model.predict_proba(X_test)
    full_metrics = evaluate_metrics(y_test, y_pred_full, y_proba_full, n_classes)

    metric_store = {
        "Accuracy": [],
        "Macro_F1": [],
        "Log_Loss": []
    }

    for b in range(N_BOOTSTRAP):
        idx = stratified_bootstrap_indices(y_test, rng)

        y_b = y_test[idx]
        pred_b = y_pred_full[idx]
        proba_b = y_proba_full[idx]

        m = evaluate_metrics(y_b, pred_b, proba_b, n_classes)

        for metric_name, value in m.items():
            metric_store[metric_name].append(value)

        bootstrap_rows.append({
            "Trait": trait_name,
            "Best_Model": best_name,
            "Bootstrap_ID": b + 1,
            "Accuracy": m["Accuracy"],
            "Macro_F1": m["Macro_F1"],
            "Log_Loss": m["Log_Loss"],
        })

    for metric_name, values in metric_store.items():
        values = np.array(values)
        ci_low, ci_high = confidence_interval(values, CI_LEVEL)

        summary_rows.append({
            "Trait": trait_name,
            "Best_Model": best_name,
            "Metric": metric_name,
            "Full_Test_Value": full_metrics[metric_name],
            "Bootstrap_Mean": values.mean(),
            "CI_Lower": ci_low,
            "CI_Upper": ci_high,
            "CI_Level": CI_LEVEL,
            "N_Bootstrap": N_BOOTSTRAP,
        })


ci_bootstrap_df = pd.DataFrame(bootstrap_rows)
ci_summary_df = pd.DataFrame(summary_rows)

ci_summary_df["Paper_Format"] = ci_summary_df.apply(
    lambda r: f"{r['Full_Test_Value']:.4f} "
              f"(95% CI: {r['CI_Lower']:.4f}–{r['CI_Upper']:.4f})",
    axis=1
)

print("=" * 80)
print("CONFIDENCE INTERVAL SUMMARY — FINAL SELECTED MODELS")
print("=" * 80)

display(ci_summary_df)

print("\nPaper-ready lines:")
for _, r in ci_summary_df.iterrows():
    print(
        f"{r['Trait']} | {r['Best_Model']} | {r['Metric']}: "
        f"{r['Paper_Format']}"
    )

In [ ]:
for label, res, choice in [("Eye", eye_res, eye_choice),
                            ("Hair", hair_res, hair_choice),
                            ("Skin", skin_res, skin_choice)]:
    y_test = res["y_test"]
    y_pred = res["predictions"][choice]
    classes = res["classes"]
    print(f"\n{'='*50}")
    print(f"  {label} Color  |  Best: {choice}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred,
                                target_names=[str(c) for c in classes], zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

In [ ]:
def entropy(probs):
    return float(-np.sum(probs * np.log(probs + 1e-9)))

def uncertainty_level(ent):
    if ent < 0.3:
        return "Very High Confidence"
    elif ent < 0.7:
        return "Moderate Confidence"
    else:
        return "Low Confidence"

In [ ]:
def predict_multitask_from_csv(csv_path):
    df_in = pd.read_csv(csv_path)

    snps_in = [c for c in selected_snps if c in df_in.columns]
    for group_name, rs_ids in gene_groups.items():
        cols = [s for s in snps_in if any(rs in s for rs in rs_ids)]
        if len(cols) >= 2:
            df_in[f"{group_name}_sum"]  = df_in[cols].sum(axis=1)
            df_in[f"{group_name}_mean"] = df_in[cols].mean(axis=1)
    if "MC1R_sum" in df_in.columns and "HERC2_OCA2_sum" in df_in.columns:
        df_in["MC1R_x_HERC2"] = df_in["MC1R_sum"] * df_in["HERC2_OCA2_sum"]
    if "MC1R_sum" in df_in.columns and "SLC_sum" in df_in.columns:
        df_in["MC1R_x_SLC"] = df_in["MC1R_sum"] * df_in["SLC_sum"]

    all_needed = set(eye_res["feature_cols"]) | set(hair_res["feature_cols"]) | set(skin_res["feature_cols"])
    missing = [c for c in all_needed if c not in df_in.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    outputs = []
    for i in range(len(df_in)):
        out = {}
        for trait_key, res, final_model in [
            ("Eye",  eye_res,  eye_final_model),
            ("Hair", hair_res, hair_final_model),
            ("Skin", skin_res, skin_final_model),
        ]:
            feat_cols = res["feature_cols"]
            X_i = df_in[feat_cols].iloc[i].values.reshape(1, -1)
            proba = final_model.predict_proba(X_i)[0]
            cls = res["classes"]
            top_idx = int(np.argmax(proba))
            out[trait_key] = {
                "Top": str(cls[top_idx]),
                "Confidence_%": round(float(np.max(proba)) * 100, 2),
                "Entropy": round(entropy(proba), 4),
                "Uncertainty": uncertainty_level(entropy(proba)),
                "Probabilities_%": {str(c): round(float(v)*100, 2) for c, v in zip(cls, proba)},
            }

        out["Summary"] = {k: out[k]["Top"] for k in ["Eye", "Hair", "Skin"]}
        out["Overall_Confidence_%"] = round(
            np.mean([out[k]["Confidence_%"] for k in ["Eye", "Hair", "Skin"]]), 2)
        outputs.append(out)
    return outputs

In [ ]:
import json as _json, os

CSV_PATH = "4_5805438016180723207.csv"
if os.path.isfile(CSV_PATH):
    results = predict_multitask_from_csv(CSV_PATH)
    print("="*60)
    print("INFERENCE RESULTS")
    print("="*60)
    print(_json.dumps(results[0], indent=2, ensure_ascii=False))
else:
    print(f"{CSV_PATH} not found; skip inference.")

## Feature Importance Plot

In [ ]:
import matplotlib.pyplot as plt

n_traits = len(targets_fs)
fig, axes = plt.subplots(1, n_traits, figsize=(18, 6))
if n_traits == 1:
    axes = [axes]

for ax, (label, target_col) in zip(axes, targets_fs.items()):
    n = target_n_features[label]
    X_fs = df[all_feature_cols]
    y_fs = df[target_col].astype(str)
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_fs, y_fs)
    importances = pd.Series(rf.feature_importances_, index=all_feature_cols)
    top_n = importances.nlargest(n).sort_values()
    top_n.plot(kind="barh", ax=ax)
    ax.set_title(f"{label} - Top {n} Features")
    ax.set_xlabel("Importance")

plt.tight_layout()
plt.show()

## [NEW] All Models Comparison (Bar Chart)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (label, res) in zip(axes, [("Eye", eye_res), ("Hair", hair_res), ("Skin", skin_res)]):
    names = list(res["metrics"].keys())
    f1s   = [res["metrics"][n]["Macro_F1"] for n in names]
    colors = ["#2ecc71" if n == choose_best_model(res)[0] else "#3498db" for n in names]
    ax.barh(names, f1s, color=colors)
    ax.set_xlim(0, 1.05)
    ax.set_title(f"{label} Color - Macro F1")
    ax.set_xlabel("Macro F1")
    for i, v in enumerate(f1s):
        ax.text(v + 0.01, i, f"{v:.4f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════
# 📊  Comprehensive Visual Analysis for Research Paper
# ═══════════════════════════════════════════════════

> **Purpose:** Generate publication-quality figures covering data exploration,
> model comparison, error analysis, and calibration — ready for a research paper.

In [ ]:

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize
from itertools import cycle
import textwrap

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 15,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

MODEL_COLORS = {
    'LogReg':   '#636EFA',
    'LGBM':     '#00CC96',
    'XGB':      '#EF553B',
    'HistGB':   '#AB63FA',
    'SVC':      '#FFA15A',
    'RF':       '#19D3F3',
    'TabNet':   '#FF6692',
    'Stacking': '#B6E880',
}
TRAIT_COLORS = {'Eye': '#636EFA', 'Hair': '#EF553B', 'Skin': '#00CC96'}
TRAIT_CMAPS  = {'Eye': 'Blues', 'Hair': 'Oranges', 'Skin': 'Greens'}

trait_results = {'Eye': eye_res, 'Hair': hair_res, 'Skin': skin_res}
model_names = list(eye_res['metrics'].keys())

print("Style configured. Ready to plot.")

## 2. Dataset Overview & Class Distribution

Visual exploration of the phenotype label distributions, showing potential class
imbalance before any resampling.

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

trait_targets = {
    'Eye Color':  EYE_TARGET,
    'Hair Color': HAIR_TARGET,
    'Skin Color': SKIN_TARGET_MERGED,
}
palette_list = ['#636EFA', '#EF553B', '#00CC96']

for ax, (label, col), color in zip(axes, trait_targets.items(), palette_list):
    counts = df[col].value_counts().sort_index()
    bars = ax.bar(counts.index, counts.values, color=color, edgecolor='white', linewidth=0.8, alpha=0.85)
    ax.set_title(f'{label} Distribution', fontweight='bold')
    ax.set_ylabel('Number of Samples')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Figure 1 — Phenotype Class Distributions (Before SMOTE)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. SNP Feature Correlation Heatmap

Pearson correlation among the top selected SNP + gene-group features. High
correlations may indicate linked loci or redundant features, informing
feature engineering decisions.

In [ ]:

all_top = sorted(set(
    trait_feature_cols['Eye Color'] +
    trait_feature_cols['Hair Color'] +
    trait_feature_cols['Skin Color']
))

corr = df[all_top].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, cmap='RdBu_r', center=0,
    vmin=-1, vmax=1, linewidths=0.3,
    square=True, ax=ax,
    cbar_kws={'shrink': 0.7, 'label': 'Pearson r'}
)
ax.set_title('Figure 2 — Feature Correlation Matrix (Union of Top Features)', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 4. Model Performance Comparison

Side-by-side bar charts comparing **Accuracy** and **Macro F1-score** for every
model on each phenotype trait. The best model per trait is highlighted.

In [ ]:

fig, axes = plt.subplots(3, 2, figsize=(16, 14))

for row, (trait, res) in enumerate(trait_results.items()):
    names = list(res['metrics'].keys())
    accs  = [res['metrics'][n]['Accuracy'] for n in names]
    f1s   = [res['metrics'][n]['Macro_F1'] for n in names]

    best_name = choose_best_model(res)[0]
    bar_colors = [MODEL_COLORS.get(n, '#999') for n in names]

    ax_acc = axes[row, 0]
    bars = ax_acc.barh(names, accs, color=bar_colors, edgecolor='white', height=0.65)
    ax_acc.set_xlim(0, 1.08)
    ax_acc.set_title(f'{trait} Color — Accuracy', fontweight='bold')
    ax_acc.set_xlabel('Accuracy')
    for i, (v, n) in enumerate(zip(accs, names)):
        marker = ' ★' if n == best_name else ''
        ax_acc.text(v + 0.01, i, f'{v:.4f}{marker}', va='center', fontsize=9)

    ax_f1 = axes[row, 1]
    bars = ax_f1.barh(names, f1s, color=bar_colors, edgecolor='white', height=0.65)
    ax_f1.set_xlim(0, 1.08)
    ax_f1.set_title(f'{trait} Color — Macro F1', fontweight='bold')
    ax_f1.set_xlabel('Macro F1')
    for i, (v, n) in enumerate(zip(f1s, names)):
        marker = ' ★' if n == best_name else ''
        ax_f1.text(v + 0.01, i, f'{v:.4f}{marker}', va='center', fontsize=9)

plt.suptitle('Figure 3 — Model Comparison: Accuracy & Macro F1 per Trait\n(★ = Best Model)',
             fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Calibration Quality — Log-Loss Comparison

Log-loss measures how well-calibrated the predicted probabilities are.
Lower is better. This is critical for forensic applications where
confidence scores matter as much as the predicted class.

In [ ]:

fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(model_names))
width = 0.25

for i, (trait, res) in enumerate(trait_results.items()):
    losses = [res['metrics'][n]['Log_Loss'] for n in model_names]
    ax.bar(x + i * width, losses, width, label=f'{trait} Color',
           color=TRAIT_COLORS[trait], edgecolor='white', alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(model_names, rotation=30, ha='right')
ax.set_ylabel('Log-Loss (lower is better)')
ax.set_title('Figure 4 — Log-Loss Comparison Across Models & Traitss', fontweight='bold')
ax.legend(frameon=False)
ax.axhline(y=0, color='grey', linewidth=0.5)
plt.tight_layout()
plt.show()

## 6. Radar Chart — Multi-Metric Model Profile

Radar (spider) charts show each model's profile across Accuracy, Macro-F1,
and (1 − Log-Loss) simultaneously, letting readers quickly judge trade-offs.

In [ ]:

from math import pi

metrics_radar = ['Accuracy', 'Macro_F1', '1-LogLoss']
n_metrics = len(metrics_radar)
angles = [n / float(n_metrics) * 2 * pi for n in range(n_metrics)]
angles += angles[:1]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), subplot_kw=dict(polar=True))

for ax, (trait, res) in zip(axes, trait_results.items()):
    ax.set_theta_offset(pi / 2)
    ax.set_theta_direction(-1)
    ax.set_rlabel_position(0)
    ax.set_title(f'{trait} Color', fontweight='bold', pad=20)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics_radar)
    ax.set_ylim(0, 1.1)

    for name in model_names:
        m = res['metrics'][name]
        values = [m['Accuracy'], m['Macro_F1'], max(0, 1 - m['Log_Loss'])]
        values += values[:1]
        ax.plot(angles, values, linewidth=1.5, label=name,
                color=MODEL_COLORS.get(name, '#999'))
        ax.fill(angles, values, alpha=0.05, color=MODEL_COLORS.get(name, '#999'))

axes[-1].legend(loc='upper right', bbox_to_anchor=(1.45, 1.1), frameon=False)
plt.suptitle('Figure 5 — Radar Chart: Multi-Metric Model Profiles', fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 7. Confusion Matrices (Best Model per Trait)

Normalised confusion matrices for the best-performing model on each trait.
Diagonal cells show correct classification rate; off-diagonal cells expose
systematic misclassifications (e.g., Dark ↔ DarkToBlack confusion).

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (trait, res) in zip(axes, trait_results.items()):
    best_name = choose_best_model(res)[0]
    y_true = res['y_test']
    y_pred = res['predictions'][best_name]
    classes = res['classes']

    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap=TRAIT_CMAPS[trait],
                xticklabels=classes, yticklabels=classes,
                linewidths=0.5, ax=ax, vmin=0, vmax=1,
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{trait} Color — {best_name}\n(Row-Normalised)', fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    ax.tick_params(axis='x', rotation=35)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Figure 6 — Normalised Confusion Matrices (Best Model per Trait)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (trait, res) in zip(axes, trait_results.items()):
    best_name = choose_best_model(res)[0]
    y_true = res['y_test']
    y_pred = res['predictions'][best_name]
    classes = res['classes']

    cm = confusion_matrix(y_true, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap=TRAIT_CMAPS[trait],
                xticklabels=classes, yticklabels=classes,
                linewidths=0.5, ax=ax,
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{trait} Color — {best_name}\n(Absolute Counts)', fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    ax.tick_params(axis='x', rotation=35)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Figure 6b — Confusion Matrices: Absolute Counts', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 8. Per-Class F1 Heatmap

A heatmap showing the F1-score of every model for every class within each
trait. This reveals which classes are hardest to predict and which models
handle minority classes best.

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

for ax, (trait, res) in zip(axes, trait_results.items()):
    classes = list(res['classes'])
    data = []
    for name in model_names:
        y_true = res['y_test']
        y_pred = res['predictions'][name]
        report = classification_report(y_true, y_pred, target_names=classes,
                                       output_dict=True, zero_division=0)
        row = [report[c]['f1-score'] for c in classes]
        data.append(row)

    df_heat = pd.DataFrame(data, index=model_names, columns=classes)
    sns.heatmap(df_heat, annot=True, fmt='.2f', cmap='YlOrRd',
                linewidths=0.5, ax=ax, vmin=0, vmax=1,
                cbar_kws={'shrink': 0.8, 'label': 'F1'})
    ax.set_title(f'{trait} Color — Per-Class F1', fontweight='bold')
    ax.set_ylabel('Model')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35)

plt.suptitle('Figure 7 — Per-Class F1-Score Heatmap (All Models × All Classes)',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Precision–Recall Curves (One-vs-Rest)

Precision–Recall curves per class for the best model on each trait.
These are more informative than ROC curves when classes are imbalanced,
which is the case for minority pigmentation phenotypes.

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

color_cycle = plt.cm.tab10.colors

for ax, (trait, res) in zip(axes, trait_results.items()):
    best_name = choose_best_model(res)[0]
    best_model = res['models'][best_name]
    classes = res['classes']
    n_cls = len(classes)
    y_true = res['y_test']

    le = res['label_encoder']
    feat_cols = res['feature_cols']
    y_all = le.transform(df[trait_targets[f'{trait} Color']].astype(str))
    X_all = df[feat_cols].values
    X_tv, X_test_full, y_tv, y_test_full = train_test_split(
        X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

    y_proba = best_model.predict_proba(X_test_full)
    y_bin = label_binarize(y_test_full, classes=list(range(n_cls)))

    for i, color in zip(range(n_cls), cycle(color_cycle)):
        if y_bin.shape[1] > i:
            precision, recall, _ = precision_recall_curve(y_bin[:, i], y_proba[:, i])
            ap = average_precision_score(y_bin[:, i], y_proba[:, i])
            ax.plot(recall, precision, color=color, lw=1.8,
                    label=f'{classes[i]} (AP={ap:.2f})')

    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'{trait} Color — {best_name}', fontweight='bold')
    ax.legend(fontsize=8, frameon=False, loc='lower left')
    ax.set_xlim([0, 1.05])
    ax.set_ylim([0, 1.05])

plt.suptitle('Figure 8 — Precision–Recall Curves (Best Model, One-vs-Rest)',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. ROC Curves (One-vs-Rest)

Receiver Operating Characteristic curves with AUC per class, providing
another view of discriminative power complementary to Precision–Recall.

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (trait, res) in zip(axes, trait_results.items()):
    best_name = choose_best_model(res)[0]
    best_model = res['models'][best_name]
    classes = res['classes']
    n_cls = len(classes)

    le = res['label_encoder']
    feat_cols = res['feature_cols']
    y_all = le.transform(df[trait_targets[f'{trait} Color']].astype(str))
    X_all = df[feat_cols].values
    X_tv, X_test_full, y_tv, y_test_full = train_test_split(
        X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

    y_proba = best_model.predict_proba(X_test_full)
    y_bin = label_binarize(y_test_full, classes=list(range(n_cls)))

    for i, color in zip(range(n_cls), cycle(color_cycle)):
        if y_bin.shape[1] > i:
            fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=color, lw=1.8,
                    label=f'{classes[i]} (AUC={roc_auc:.2f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'{trait} Color — {best_name}', fontweight='bold')
    ax.legend(fontsize=8, frameon=False, loc='lower right')
    ax.set_xlim([0, 1.02])
    ax.set_ylim([0, 1.05])

plt.suptitle('Figure 9 — ROC Curves (Best Model, One-vs-Rest)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11. Feature Importance — Top 15 per Trait

Horizontal bar charts showing the most predictive SNPs/gene-group features.
This is key for biological interpretation and validates known pigmentation loci
(HERC2, OCA2, MC1R, SLC24A5, etc.).

In [ ]:

gene_color_map = {
    'MC1R': '#EF553B', 'HERC2_OCA2': '#636EFA',
    'SLC': '#00CC96', 'TYR': '#AB63FA',
}

def get_gene_color(feat):
    for group in gene_color_map:
        if group in feat:
            return gene_color_map[group]
    return '#888888'

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, (label, target_col) in zip(axes, trait_targets.items()):
    X_fs = df[all_feature_cols]
    y_fs = df[target_col].astype(str)
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_fs, y_fs)
    importances = pd.Series(rf.feature_importances_, index=all_feature_cols)
    top15 = importances.nlargest(15).sort_values()
    colors = [get_gene_color(f) for f in top15.index]
    top15.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.set_title(f'{label} — Top 15 Features', fontweight='bold')
    ax.set_xlabel('Gini Importance')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=g) for g, c in gene_color_map.items()]
legend_elements.append(Patch(facecolor='#888888', label='Other SNP'))
axes[-1].legend(handles=legend_elements, loc='lower right', fontsize=8, frameon=False)

plt.suptitle('Figure 10 — Top 15 Feature Importances (Colour = Gene Group)',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 12. Comprehensive Results Summary Table

A single table collecting all metrics for all models across all three
traits — perfect for inclusion as **Table 1** in the paper.

In [ ]:

rows = []
for trait, res in trait_results.items():
    best_name = choose_best_model(res)[0]
    for name in model_names:
        m = res['metrics'][name]
        rows.append({
            'Trait': f'{trait} Color',
            'Model': name,
            'Accuracy': m['Accuracy'],
            'Macro F1': m['Macro_F1'],
            'Log-Loss': m['Log_Loss'],
            'Best?': '★' if name == best_name else '',
        })

summary_df = pd.DataFrame(rows)

def highlight_best(row):
    if row['Best?'] == '★':
        return ['background-color: #d4edda; font-weight: bold'] * len(row)
    return [''] * len(row)

styled = (summary_df.style
    .apply(highlight_best, axis=1)
    .format({'Accuracy': '{:.4f}', 'Macro F1': '{:.4f}', 'Log-Loss': '{:.4f}'})
    .set_caption('Table 1 — Complete Model Performance Summary')
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '14px')]},
    ])
)
display(styled)

print("\n--- LaTeX Table ---")
print(summary_df.to_latex(index=False, float_format='%.4f'))

## 13. Metric Heatmap Grid

An alternative visual encoding: a heatmap where rows are models, columns
are trait×metric combinations. This provides a compact bird's-eye view.

In [ ]:

data_acc, data_f1, data_ll = {}, {}, {}
for trait, res in trait_results.items():
    data_acc[trait] = [res['metrics'][n]['Accuracy'] for n in model_names]
    data_f1[trait]  = [res['metrics'][n]['Macro_F1'] for n in model_names]
    data_ll[trait]  = [res['metrics'][n]['Log_Loss'] for n in model_names]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (metric_name, data_dict, cmap, vmin, vmax) in zip(axes, [
    ('Accuracy', data_acc, 'YlGn', 0.4, 1.0),
    ('Macro F1', data_f1,  'YlOrRd', 0.0, 1.0),
    ('Log-Loss', data_ll,  'YlOrRd_r', 0.0, 3.0),
]):
    df_grid = pd.DataFrame(data_dict, index=model_names)
    df_grid.columns = [f'{c} Color' for c in df_grid.columns]
    sns.heatmap(df_grid, annot=True, fmt='.3f', cmap=cmap,
                vmin=vmin, vmax=vmax, linewidths=0.5, ax=ax,
                cbar_kws={'shrink': 0.8})
    ax.set_title(f'{metric_name}', fontweight='bold')
    ax.set_ylabel('')

plt.suptitle('Figure 11 — Metric Heatmap Grid (Models × Traits)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 14. Optuna Hyperparameter Summary

Displays the best hyperparameters found by Bayesian optimisation (Optuna)
for LightGBM and XGBoost on each trait — important for reproducibility.

In [ ]:

for trait, res in trait_results.items():
    print(f"\n{'='*55}")
    print(f"  {trait} Color — Optuna Best Params")
    print(f"{'='*55}")
    for algo in ['lgbm', 'xgb']:
        print(f"\n  [{algo.upper()}]")
        for k, v in res['optuna_params'][algo].items():
            print(f"    {k:25s} = {v}")

## 15. Final Model Selection Recap

A concise visual showing which model was chosen as the winner for each
trait and its headline metrics.

In [ ]:

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

table_data = []
for trait, res in trait_results.items():
    best_name = choose_best_model(res)[0]
    m = res['metrics'][best_name]
    table_data.append([
        f'{trait} Color',
        best_name,
        f"{m['Accuracy']:.4f}",
        f"{m['Macro_F1']:.4f}",
        f"{m['Log_Loss']:.4f}",
    ])

table = ax.table(
    cellText=table_data,
    colLabels=['Trait', 'Best Model', 'Accuracy', 'Macro F1', 'Log-Loss'],
    cellLoc='center', loc='center',
    colColours=['#636EFA'] * 5,
)
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2.0)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_edgecolor('#ddd')

ax.set_title('Table 2 — Final Selected Models & Performance',
             fontweight='bold', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("\n✅ All analysis figures generated successfully!")
print("   Save individual figures with: fig.savefig('figure_N.png', dpi=300, bbox_inches='tight')")

In [ ]:

import os
import matplotlib.pyplot as plt

OUTPUT_DIR = "confidence_interval_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ci_summary_df.to_csv(f"{OUTPUT_DIR}/confidence_interval_summary.csv", index=False)
ci_bootstrap_df.to_csv(f"{OUTPUT_DIR}/confidence_interval_bootstrap_values.csv", index=False)


plot_df = ci_summary_df[ci_summary_df["Metric"].isin(["Accuracy", "Macro_F1"])].copy()

traits = ["Eye", "Hair", "Skin"]
metrics = ["Accuracy", "Macro_F1"]

x = np.arange(len(traits))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

for i, metric in enumerate(metrics):
    metric_df = plot_df[plot_df["Metric"] == metric].set_index("Trait").loc[traits]

    y = metric_df["Full_Test_Value"].values
    ci_low = metric_df["CI_Lower"].values
    ci_high = metric_df["CI_Upper"].values

    lower_err = np.maximum(0, y - ci_low)
    upper_err = np.maximum(0, ci_high - y)

    offset = (i - 0.5) * width

    bars = ax.bar(
        x + offset,
        y,
        width,
        yerr=[lower_err, upper_err],
        capsize=6,
        label=metric.replace("_", " "),
        alpha=0.85
    )

    for bar, value in zip(bars, y):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

ax.set_title(
    "Final Selected Models Performance with 95% Confidence Intervals\n"
    f"Based on {N_BOOTSTRAP} Stratified Bootstrap Test Resamples",
    fontsize=13,
    fontweight="bold"
)
ax.set_ylabel("Score")
ax.set_xticks(x)
ax.set_xticklabels([f"{t} Color" for t in traits])
ax.set_ylim(0, 1.08)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.35)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/CI_accuracy_macro_f1.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/CI_accuracy_macro_f1.pdf", bbox_inches="tight")
plt.show()



loss_df = ci_summary_df[ci_summary_df["Metric"] == "Log_Loss"].set_index("Trait").loc[traits]

y = loss_df["Full_Test_Value"].values
ci_low = loss_df["CI_Lower"].values
ci_high = loss_df["CI_Upper"].values

lower_err = np.maximum(0, y - ci_low)
upper_err = np.maximum(0, ci_high - y)

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    traits,
    y,
    yerr=[lower_err, upper_err],
    capsize=6,
    alpha=0.85
)

for bar, value in zip(bars, y):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

ax.set_title(
    "Log-Loss with 95% Confidence Intervals\n"
    f"Based on {N_BOOTSTRAP} Stratified Bootstrap Test Resamples",
    fontsize=13,
    fontweight="bold"
)
ax.set_ylabel("Log-Loss")
ax.set_xlabel("Trait")
ax.grid(axis="y", linestyle="--", alpha=0.35)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/CI_log_loss.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/CI_log_loss.pdf", bbox_inches="tight")
plt.show()



paper_table = ci_summary_df.pivot_table(
    index=["Trait", "Best_Model"],
    columns="Metric",
    values="Paper_Format",
    aggfunc="first"
).reset_index()

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.axis("off")

table = ax.table(
    cellText=paper_table.values,
    colLabels=paper_table.columns,
    cellLoc="center",
    loc="center"
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.15, 1.8)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight="bold", color="white")
        cell.set_facecolor("#40466e")
    else:
        cell.set_edgecolor("#dddddd")

ax.set_title(
    "Table — Final Selected Models with 95% Confidence Intervals",
    fontsize=13,
    fontweight="bold",
    pad=15
)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/CI_summary_table.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUTPUT_DIR}/CI_summary_table.pdf", bbox_inches="tight")
plt.show()

print("Saved outputs:")
print(f"- {OUTPUT_DIR}/confidence_interval_summary.csv")
print(f"- {OUTPUT_DIR}/confidence_interval_bootstrap_values.csv")
print(f"- {OUTPUT_DIR}/CI_accuracy_macro_f1.png")
print(f"- {OUTPUT_DIR}/CI_log_loss.png")
print(f"- {OUTPUT_DIR}/CI_summary_table.png")